This file currently works.  Please save this as a backup.

In [1]:
# =============================
# Imports
# =============================
import os
import json
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# =============================
# Configuration
# =============================
OUTPUT_FOLDER = "data"
RAW_COMBINED_CSV = os.path.join(OUTPUT_FOLDER, "etl-data-raw.csv")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# List of tickers to fetch
etf_list = ["SPLG","XLG","TOPT","QQQ","VGT","QTOP","FBCG","MSFT","GOOGL","UPRO","TQQQ","QQUP","GGLL","MSFU","OEF","QQQJ","VTI","ALLY","HSBC","ARKK","FMAG","QQXL","AMZN","MGK","AAPL","NVDA"]


In [2]:
#def safe_str_date(dt):
#    if pd.isna(dt):
#        return ""
#    if hasattr(dt, "strftime"):
#        return dt.strftime("%Y-%m-%d")
#    return str(dt)

In [3]:
rows = []
for sym in etf_list:
    print(f"[fetch] {sym}")
    try:
        t = yf.Ticker(sym)
        df = t.history(period="20y", interval="1d", auto_adjust=True)
        if df is None or df.empty:
            print(f"  no data for {sym}, skipping")
            continue

        df = df.reset_index()
        df["Symbol"] = sym

        keep_cols = ["Symbol", "Date", "Open", "High", "Low", "Close", "Volume"]
        for c in keep_cols:
            if c not in df.columns:
                df[c] = pd.NA
        df = df[keep_cols]

        rows.append(df)
    except Exception as e:
        print(f"  error fetching {sym}: {e}")

if rows:
    combined = pd.concat(rows, ignore_index=True)
    combined.to_csv(RAW_COMBINED_CSV, index=False)
    print(f"Wrote raw combined CSV -> {RAW_COMBINED_CSV}")
else:
    print("No data downloaded.")


[fetch] SPLG
[fetch] XLG
[fetch] TOPT
[fetch] QQQ
[fetch] VGT
[fetch] QTOP
[fetch] FBCG
[fetch] MSFT
[fetch] GOOGL
[fetch] UPRO
[fetch] TQQQ
[fetch] QQUP
[fetch] GGLL
[fetch] MSFU
[fetch] OEF
[fetch] QQQJ
[fetch] VTI
[fetch] ALLY
[fetch] HSBC
[fetch] ARKK
[fetch] FMAG
[fetch] QQXL
[fetch] AMZN
[fetch] MGK
[fetch] AAPL
[fetch] NVDA
Wrote raw combined CSV -> data\etl-data-raw.csv


In [4]:
from datetime import datetime, timedelta

# Load raw CSV
df = pd.read_csv(RAW_COMBINED_CSV)

# Ensure Date is parsed as datetime (use utc=True to avoid mixed timezone FutureWarning)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)

def extract_date_components(date_val):
    if pd.isna(date_val):
        return {'year': None, 'month': None, 'week_of_year': None, 'weekday': None}
    
    # Convert to naive datetime (drop timezone info if present)
    if hasattr(date_val, 'to_pydatetime'):
        dt = date_val.to_pydatetime()
        if dt.tzinfo is not None:
            dt = dt.tz_convert(None) if hasattr(dt, 'tz_convert') else dt.replace(tzinfo=None)
    else:
        dt = date_val
    
    # If dt is pandas Timestamp with tzinfo, convert to naive by normalizing to date in local time
    try:
        if getattr(dt, 'tzinfo', None) is not None:
            dt = dt.tz_convert(None) if hasattr(dt, 'tz_convert') else dt.replace(tzinfo=None)
    except Exception:
        pass

    # First Monday of the year
    jan_1 = datetime(dt.year, 1, 1)
    if jan_1.weekday() == 0:  # Monday
        first_monday = jan_1
    else:
        days_to_monday = 7 - jan_1.weekday()
        first_monday = jan_1 + timedelta(days=days_to_monday)

    # Financial week (1–52)
    if dt >= first_monday:
        financial_week = min(52, ((dt - first_monday).days // 7) + 1)
    else:
        financial_week = 1

    return {
        "year": dt.year,
        "month": dt.month,
        "week_of_year": financial_week,
        "weekday": dt.strftime("%A"),
    }

# --- Apply date components ---
date_components = df["Date"].apply(extract_date_components)
df["Year"] = [comp["year"] for comp in date_components]
df["Month"] = [comp["month"] for comp in date_components]
df["Week"] = [comp["week_of_year"] for comp in date_components]
df["Weekday"] = [comp["weekday"] for comp in date_components]

# --- Numeric conversions ---
df[["Open", "High", "Low", "Close"]] = df[["Open", "High", "Low", "Close"]].apply(pd.to_numeric, errors="coerce")

# --- Avg daily price ---
df["avg_daily_price"] = df[["Open", "High", "Low", "Close"]].mean(axis=1).round(4)

# --- Previous close per symbol ---
df = df.sort_values(["Symbol", "Date"])
df["Previous_Close"] = df.groupby("Symbol")["Close"].shift(1)

# --- Percent metrics ---
mask = df["Previous_Close"].notna() & (df["Previous_Close"] != 0)
if mask.any():
    prev = df.loc[mask, "Previous_Close"].astype(float)

    df.loc[mask, "Daily_Gain_Loss_Pct"] = ((df.loc[mask, "Close"] - prev) / prev * 100).round(2)
    df.loc[mask, "Open_vs_PrevClose_Pct"] = ((df.loc[mask, "Open"] - prev) / prev * 100).round(2)
    df.loc[mask, "Low_vs_PrevClose_Pct"] = ((df.loc[mask, "Low"] - prev) / prev * 100).round(2)
    df.loc[mask, "Close_vs_PrevClose_Pct"] = df.loc[mask, "Daily_Gain_Loss_Pct"]
    df.loc[mask, "mx_percent_decline"] = ((prev - df.loc[mask, "Low"]) / prev * 100).round(2)





print("Transformed dataframe shape:", df.shape)
df.to_csv(os.path.join(OUTPUT_FOLDER, "etl-data-processed.csv"), index=False)

Transformed dataframe shape: (84128, 18)


In [5]:
# Load processed daily history (from prior cell output)
proc_path = os.path.join(OUTPUT_FOLDER, "etl-data-processed.csv")
df = pd.read_csv(proc_path)

# Parse and ensure numeric types
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df[["Previous_Close", "Low"]] = df[["Previous_Close", "Low"]].apply(pd.to_numeric, errors="coerce")

# --- Define BOD levels as factor multipliers (Previous_Close * factor) ---
bod_levels = {
    "BOD99-1": 0.99,
    "BOD98-2": 0.98,
    "BOD97-3": 0.97,
    "BOD96-4": 0.96,
    "BOD95-5": 0.95,
    "BOD94-6": 0.94,
    "BOD93-7": 0.93,
    "BOD92-8": 0.92,
    "BOD91-9": 0.91,
    "BOD90-10": 0.90,
    "BOD89-11": 0.89,
    "BOD88-12": 0.88,
    "BOD87-13": 0.87,
    "BOD86-14": 0.86,
    "BOD85-15": 0.85,
    "BOD84-16": 0.84,
    "BOD82-18": 0.82,
    "BOD81-19": 0.81,
    "BOD80-20": 0.80,
}

import numpy as np
cols = list(bod_levels.keys())
factors = np.array(list(bod_levels.values()), dtype=float)

# Vectorized computation of limit prices (rows x levels)
prev = df['Previous_Close'].to_numpy(dtype=float)  # NaN where unavailable
low = df['Low'].to_numpy(dtype=float)
limits = (prev[:, None] * factors[None, :])  # shape (n_rows, n_levels)
limits_rounded = np.round(limits, 4)

# Write per-level limit price columns into dataframe
for i, col in enumerate(cols):
    df[col] = limits_rounded[:, i]

# Executed when: low <= limit_price <= previous_close (and prev/low are numeric)
executed_mask = (~np.isnan(prev))[:, None] & (~np.isnan(low))[:, None] & (limits >= low[:, None]) & (limits <= prev[:, None])

# Compute shares purchased (1 share per executed bucket) and total invested (sum of executed limit prices)
shares = executed_mask.sum(axis=1).astype(int)
total_value = np.round((executed_mask * limits).sum(axis=1), 4)
df['Shares_Purchased'] = shares
df['Total_Value_Purchased'] = total_value

# Compute Executed_Levels as comma-separated list per row
def _levels_from_mask(mask_row, level_names):
    names = [level_names[i] for i, v in enumerate(mask_row) if v]
    return ",".join(names) if names else ""

executed_levels_list = [_levels_from_mask(row_mask, cols) for row_mask in executed_mask]
df['Executed_Levels'] = executed_levels_list

# Add per-level boolean executed columns (e.g., 'BOD99-1_Executed')
for i, col in enumerate(cols):
    exec_col = f"{col}_Executed"
    df[exec_col] = executed_mask[:, i]
    # ensure dtype bool for CSV clarity
    df[exec_col] = df[exec_col].astype(bool)

# Compute cumulative totals per symbol (ordered by date)
df = df.sort_values(['Symbol', 'Date'])
df['Cumulative_Shares'] = df.groupby('Symbol')['Shares_Purchased'].cumsum()
df['Cumulative_Invested'] = df.groupby('Symbol')['Total_Value_Purchased'].cumsum().round(4)

# --- Save BOD-enhanced file ---
bod_path = os.path.join(OUTPUT_FOLDER, "etl-data-bod.csv")
df.to_csv(bod_path, index=False)
print(f"Wrote BOD-enhanced data -> {bod_path}")

# --- Diagnostic: show what happened on 2025-06-13 ---
target_date = pd.to_datetime('2025-06-13').date()
rows_on_date = df[df['Date'].dt.date == target_date]
if rows_on_date.empty:
    print(f"No rows found for {target_date}")
else:
    print(f"\nRows for {target_date}:")
    # show relevant columns and the per-level executed flags for clarity
    bool_exec_cols = [f"{c}_Executed" for c in cols]
    show_cols = ['Symbol', 'Date', 'Previous_Close', 'Low', 'Shares_Purchased', 'Total_Value_Purchased', 'Executed_Levels', 'Cumulative_Shares', 'Cumulative_Invested'] + cols + bool_exec_cols
    display(rows_on_date[show_cols].head(5))

Wrote BOD-enhanced data -> data\etl-data-bod.csv

Rows for 2025-06-13:


,Symbol,Date,Previous_Close,Low,Shares_Purchased,Total_Value_Purchased,Executed_Levels,Cumulative_Shares,Cumulative_Invested,BOD99-1,...,BOD90-10_Executed,BOD89-11_Executed,BOD88-12_Executed,BOD87-13_Executed,BOD86-14_Executed,BOD85-15_Executed,BOD84-16_Executed,BOD82-18_Executed,BOD81-19_Executed,BOD80-20_Executed
4971,AAPL,2025-06-13 04:00:00+00:00,198.974182,195.478145,1,196.9844,BOD99-1,4182,216930.3133,196.9844,...,False,False,False,False,False,False,False,False,False,False
7892,ALLY,2025-06-13 04:00:00+00:00,36.498402,35.615457,2,71.9019,"BOD99-1,BOD98-2",3011,74609.1894,36.1334,...,False,False,False,False,False,False,False,False,False,False
12922,AMZN,2025-06-13 04:00:00+00:00,213.240005,209.619995,1,211.1076,BOD99-1,5170,275270.0658,211.1076,...,False,False,False,False,False,False,False,False,False,False
15650,ARKK,2025-06-13 04:00:00+00:00,62.020000,60.540001,2,122.1794,"BOD99-1,BOD98-2",2959,162510.7946,61.3998,...,False,False,False,False,False,False,False,False,False,False
16971,FBCG,2025-06-13 04:00:00+00:00,46.240002,45.369999,1,45.7776,BOD99-1,906,27352.5167,45.7776,...,False,False,False,False,False,False,False,False,False,False


In [6]:
# Export per-ticker CSV files for UI consumption
import os
import json

# Read the BOD-enhanced CSV we just wrote
bod_path = os.path.join(OUTPUT_FOLDER, "etl-data-bod.csv")
combined = pd.read_csv(bod_path)

# Ensure avg_daily_price exists (compute if missing)
if 'avg_daily_price' not in combined.columns:
    combined[["Open", "High", "Low", "Close"]] = combined[["Open", "High", "Low", "Close"]].apply(pd.to_numeric, errors='coerce')
    combined['avg_daily_price'] = combined[["Open", "High", "Low", "Close"]].mean(axis=1).round(4)

# Create output directory for tickers
tickers_dir = os.path.join(OUTPUT_FOLDER, 'tickers')
os.makedirs(tickers_dir, exist_ok=True)

# Write one CSV per Symbol and build an index
index = {}
for sym, grp in combined.groupby('Symbol'):
    fname = f"{sym}.csv"
    path = os.path.join(tickers_dir, fname)
    grp.to_csv(path, index=False)
    index[sym] = os.path.relpath(path, start=OUTPUT_FOLDER)

# Save index.json in data/
index_path = os.path.join(OUTPUT_FOLDER, 'tickers_index.json')
with open(index_path, 'w') as f:
    json.dump(index, f, indent=2)

print(f"Wrote {len(index)} ticker files -> {tickers_dir}")
print(f"Index file -> {index_path}")

# Show sample head for the first ticker
sample_sym = next(iter(index))
sample_path = os.path.join(tickers_dir, f"{sample_sym}.csv")
print(f"\nSample head for {sample_sym}:")
print(pd.read_csv(sample_path).head().to_string())


Wrote 26 ticker files -> data\tickers
Index file -> data\tickers_index.json

Sample head for AAPL:
  Symbol                       Date      Open      High       Low     Close     Volume  Year  Month  Week    Weekday  avg_daily_price  Previous_Close  Daily_Gain_Loss_Pct  Open_vs_PrevClose_Pct  Low_vs_PrevClose_Pct  Close_vs_PrevClose_Pct  mx_percent_decline  BOD99-1  BOD98-2  BOD97-3  BOD96-4  BOD95-5  BOD94-6  BOD93-7  BOD92-8  BOD91-9  BOD90-10  BOD89-11  BOD88-12  BOD87-13  BOD86-14  BOD85-15  BOD84-16  BOD82-18  BOD81-19  BOD80-20  Shares_Purchased  Total_Value_Purchased  Executed_Levels  BOD99-1_Executed  BOD98-2_Executed  BOD97-3_Executed  BOD96-4_Executed  BOD95-5_Executed  BOD94-6_Executed  BOD93-7_Executed  BOD92-8_Executed  BOD91-9_Executed  BOD90-10_Executed  BOD89-11_Executed  BOD88-12_Executed  BOD87-13_Executed  BOD86-14_Executed  BOD85-15_Executed  BOD84-16_Executed  BOD82-18_Executed  BOD81-19_Executed  BOD80-20_Executed  Cumulative_Shares  Cumulative_Invested
0   AAPL  